In [ ]:
import torch
import transformers
import datasets
import peft
import trl
import bitsandbytes

print("Python:", __import__("sys").version)
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("\nLibraries:")
print("transformers:", transformers.__version__)
print("datasets:", datasets.__version__)
print("peft:", peft.__version__)
print("trl:", trl.__version__)
print("bitsandbytes:", bitsandbytes.__version__)

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/SupportOpsAI"

print(os.listdir(PROJECT_DIR))

In [ ]:
import pandas as pd

processed_path = f"{PROJECT_DIR}/data/processed"

train_df = pd.read_csv(f"{processed_path}/train.csv")
val_df = pd.read_csv(f"{processed_path}/validation.csv")
test_df = pd.read_csv(f"{processed_path}/test.csv")

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

print("Tokenizer loaded")
print("Pad token:", tokenizer.pad_token)

In [ ]:
from datasets import Dataset

SYSTEM_PROMPT = """You are a customer support ticket classifier.

Choose exactly one queue from:
Technical Support
Product Support
Customer Service
IT Support
Billing and Payments
Returns and Exchanges
Service Outages and Maintenance
Sales and Pre-Sales
Human Resources
General Inquiry

Choose exactly one priority from:
low
medium
high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
"""


def format_ticket(row):
    subject = row["subject"] if pd.notna(row["subject"]) else ""

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Subject: {subject}\n\n"
                    f"Body: {row['body']}"
                )
            },
            {
                "role": "assistant",
                "content": (
                    f'{{"queue": "{row["queue"]}", '
                    f'"priority": "{row["priority"]}"}}'
                )
            }
        ]
    }


train_dataset = Dataset.from_list(
    train_df.apply(format_ticket, axis=1).tolist()
)

val_dataset = Dataset.from_list(
    val_df.apply(format_ticket, axis=1).tolist()
)

test_dataset = Dataset.from_list(
    test_df.apply(format_ticket, axis=1).tolist()
)

print(train_dataset)
print(val_dataset)
print(test_dataset)

In [ ]:
def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example


train_dataset = train_dataset.map(apply_chat_template)
val_dataset = val_dataset.map(apply_chat_template)
test_dataset = test_dataset.map(apply_chat_template)

In [ ]:
print(train_dataset[0]["text"])

In [ ]:
import numpy as np

lengths = []

for example in train_dataset:
    tokens = tokenizer(
        example["text"],
        add_special_tokens=False
    )["input_ids"]

    lengths.append(len(tokens))

print("Min:", min(lengths))
print("Max:", max(lengths))
print("Average:", np.mean(lengths))

In [ ]:
# ============================================================
# SupportOpsAI — Restore training environment
# Resume from latest QLoRA checkpoint
# ============================================================

import os
import torch
import pandas as pd

from google.colab import drive

# ------------------------------------------------------------
# 1. Mount Google Drive
# ------------------------------------------------------------

drive.mount("/content/drive")

# ------------------------------------------------------------
# 2. Project paths + model
# ------------------------------------------------------------

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

processed_path = "/content/drive/MyDrive/SupportOpsAI/data/processed"
output_dir = "/content/drive/MyDrive/SupportOpsAI/models/supportopsai-qlora"

print("Model:", model_name)
print("Output:", output_dir)

# ------------------------------------------------------------
# 3. Check GPU + library versions
# ------------------------------------------------------------

import transformers
import datasets
import peft
import trl
import bitsandbytes as bnb

print("\nEnvironment:")
print("PyTorch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("bitsandbytes:", bnb.__version__)

# ------------------------------------------------------------
# 4. Load datasets
# ------------------------------------------------------------

train_df = pd.read_csv(f"{processed_path}/train.csv")
val_df = pd.read_csv(f"{processed_path}/validation.csv")

print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(val_df))

# ------------------------------------------------------------
# 5. Tokenizer
# ------------------------------------------------------------

from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name)

# ------------------------------------------------------------
# 6. System prompt
# ------------------------------------------------------------

SYSTEM_PROMPT = """You are a customer support ticket classifier.

Choose exactly one queue from:
Technical Support
Product Support
Customer Service
IT Support
Billing and Payments
Returns and Exchanges
Service Outages and Maintenance
Sales and Pre-Sales
Human Resources
General Inquiry

Choose exactly one priority from:
low
medium
high

Return ONLY valid JSON in exactly this format:
{"queue": "<queue>", "priority": "<priority>"}
"""

# ------------------------------------------------------------
# 7. Convert dataframe rows into chat examples
# ------------------------------------------------------------

from datasets import Dataset

def format_ticket(row):
    subject = row["subject"] if pd.notna(row["subject"]) else ""

    return {
        "messages": [
            {
                "role": "system",
                "content": SYSTEM_PROMPT
            },
            {
                "role": "user",
                "content": (
                    f"Subject: {subject}\n\n"
                    f"Body: {row['body']}"
                )
            },
            {
                "role": "assistant",
                "content": (
                    f'{{"queue": "{row["queue"]}", '
                    f'"priority": "{row["priority"]}"}}'
                )
            }
        ]
    }

train_dataset = Dataset.from_pandas(train_df)
val_dataset = Dataset.from_pandas(val_df)

train_dataset = train_dataset.map(
    format_ticket,
    remove_columns=train_dataset.column_names
)

val_dataset = val_dataset.map(
    format_ticket,
    remove_columns=val_dataset.column_names
)

# ------------------------------------------------------------
# 8. Apply Qwen chat template
# ------------------------------------------------------------

def apply_chat_template(example):
    example["text"] = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return example

train_dataset = train_dataset.map(apply_chat_template)
val_dataset = val_dataset.map(apply_chat_template)

print("\nFormatted datasets:")
print(train_dataset)
print(val_dataset)

# ------------------------------------------------------------
# 9. Load 4-bit QLoRA base model
# ------------------------------------------------------------

from transformers import (
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

# ------------------------------------------------------------
# 10. Prepare model for k-bit training
# ------------------------------------------------------------

from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(
    model,
    use_gradient_checkpointing=True
)

# ------------------------------------------------------------
# 11. LoRA configuration
# ------------------------------------------------------------

from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
    ],
)

model = get_peft_model(model, lora_config)

# ------------------------------------------------------------
# 12. Fix LoRA dtype
#     (Important: prevents the BF16 GradScaler error we hit)
# ------------------------------------------------------------

for name, param in model.named_parameters():
    if param.requires_grad:
        param.data = param.data.float()

# ------------------------------------------------------------
# 13. Verify trainable parameters
# ------------------------------------------------------------

print("\nTrainable parameters:")
model.print_trainable_parameters()

from collections import Counter

trainable_dtypes = Counter(
    str(param.dtype)
    for param in model.parameters()
    if param.requires_grad
)

print("Trainable dtypes:", trainable_dtypes)

# ------------------------------------------------------------
# 14. Find latest checkpoint
# ------------------------------------------------------------

from transformers.trainer_utils import get_last_checkpoint

last_checkpoint = get_last_checkpoint(output_dir)

print("\nLatest checkpoint:")
print(last_checkpoint)

if last_checkpoint is None:
    raise RuntimeError(
        "No checkpoint found! Do NOT start training."
    )

# ------------------------------------------------------------
# 15. Training configuration
# ------------------------------------------------------------

from trl import SFTConfig

training_args = SFTConfig(
    output_dir=output_dir,

    num_train_epochs=3,

    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=180,

    # Important: both disabled because FP16 GradScaler
    # caused the BF16 gradient error on the T4.
    fp16=False,
    bf16=False,

    gradient_checkpointing=True,
    use_cache=False,

    max_length=512,
    packing=False,

    eval_strategy="epoch",
    per_device_eval_batch_size=4,

    save_strategy="epoch",
    save_total_limit=2,

    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,

    logging_strategy="steps",
    logging_steps=50,

    report_to="none",
    seed=42,

    dataset_text_field="text",
)

# ------------------------------------------------------------
# 16. Create Trainer
# ------------------------------------------------------------

from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
)

print("\n================================================")
print("READY TO RESUME")
print("================================================")
print("Checkpoint:", last_checkpoint)
print("Current training steps:", trainer.state.global_step)
print("DO NOT call trainer.train() yet.")